# LeetCode 2890 — Reshape Data: Melt bằng Python (pandas)

Link đề: https://leetcode.com/problems/reshape-data-melt/ · Độ khó: **Easy** · Chủ đề: Pandas (30 Days of Pandas)

Skeleton LeetCode (không có `class Solution`; tên hàm / chữ ký: đối chiếu lại với đề gốc):

```python
import pandas as pd

def meltTable(report: pd.DataFrame) -> pd.DataFrame:
    ...
```

Ghi chú: dữ liệu ví dụ trong notebook được viết theo trí nhớ về đề; nên đối chiếu lại với đề gốc nếu muốn khớp từng ô.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: bảng `report` ở dạng **"rộng"** (mỗi sản phẩm một hàng, có 4 cột `quarter_1`..`quarter_4`); **"nung chảy" (melt)** thành dạng **"dài"** với 3 cột `product`, `quarter`, `sales`.
2. Phân biệt **cột định danh (`id_vars`)** (giữ nguyên) và **cột giá trị (`value_vars`)** (được gập xuống thành các dòng), và hiểu `melt` là phép **rộng -> dài** (ngược của `pivot`).
3. Đi từ cách **hai vòng lặp lồng nhau tạo từng dòng** → **`melt(id_vars=..., var_name=..., value_name=...)`** → **`melt` với `value_vars` tường minh** → **`set_index().stack().reset_index()`** → **`concat` từng cột con**.
4. Biết **thứ tự dòng** của `melt` (theo từng cột giá trị, rồi tới từng hàng) khác `stack` (theo từng hàng, rồi tới từng cột).
5. Tránh các lỗi kinh điển: quên `var_name`/`value_name` (ra tên `variable`, `value`), quên `id_vars` (melt luôn cả `product`), nhầm thứ tự cột kết quả, và quên đánh lại nhãn hàng.

## 2. Đề bài & ví dụ

Có một bảng `report` gồm cột `product` và bốn cột doanh số theo quý `quarter_1`, `quarter_2`, `quarter_3`, `quarter_4`.

Hãy viết hàm **biến bảng rộng này thành bảng dài** có ba cột: `product`, `quarter`, `sales`. Mỗi **ô doanh số** của bảng gốc trở thành **một dòng** của bảng mới.

**Ví dụ**

- Input: DataFrame `report`

| product | quarter_1 | quarter_2 | quarter_3 | quarter_4 |
|---|---|---|---|---|
| Umbrella | 417 | 224 | 379 | 611 |
| SleepingBag | 800 | 936 | 93 | 875 |

- Output:

| product | quarter | sales |
|---|---|---|
| Umbrella | quarter_1 | 417 |
| SleepingBag | quarter_1 | 800 |
| Umbrella | quarter_2 | 224 |
| SleepingBag | quarter_2 | 936 |
| Umbrella | quarter_3 | 379 |
| SleepingBag | quarter_3 | 93 |
| Umbrella | quarter_4 | 611 |
| SleepingBag | quarter_4 | 875 |

- Giải thích: `2` sản phẩm x `4` quý = `8` dòng. Tên các cột `quarter_i` trở thành **giá trị** của cột `quarter`, còn doanh số nằm ở cột `sales`. Thứ tự dòng: lần lượt từng quý (cột), trong mỗi quý theo thứ tự sản phẩm.

(Giá trị cụ thể trong bảng: đối chiếu lại với đề gốc.)

In [1]:
from typing import List
import pandas as pd
import numpy as np

report = pd.DataFrame({
    "product": ["Umbrella", "SleepingBag"],
    "quarter_1": [417, 800],
    "quarter_2": [224, 936],
    "quarter_3": [379, 93],
    "quarter_4": [611, 875],
})
print(report)
print()
print(report.dtypes)
print("hình dạng:", report.shape)
print("pandas", pd.__version__)

       product  quarter_1  quarter_2  quarter_3  quarter_4
0     Umbrella        417        224        379        611
1  SleepingBag        800        936         93        875

product        str
quarter_1    int64
quarter_2    int64
quarter_3    int64
quarter_4    int64
dtype: object
hình dạng: (2, 5)
pandas 3.0.5


## 3. Kiến thức nền

### 3.1. Dạng rộng (wide) và dạng dài (long)

- **Dạng rộng:** mỗi **giá trị của một biến** (ở đây: mỗi quý) là **một cột**. `report` là dạng rộng: 2 hàng, 5 cột.
- **Dạng dài:** mỗi **quan sát** (một sản phẩm trong một quý) là **một dòng**, các cột là `product`, `quarter`, `sales`. Thuận lợi để vẽ biểu đồ, nhóm, lọc.

`melt` biến **rộng -> dài**. Số dòng mới = số hàng x số cột giá trị = `2 x 4 = 8`.

In [2]:
n_rows = len(report)
value_cols = [c for c in report.columns if c != "product"]
print("cột giá trị:", value_cols)
print("số dòng của bảng dài =", n_rows, "x", len(value_cols), "=", n_rows * len(value_cols))

cột giá trị: ['quarter_1', 'quarter_2', 'quarter_3', 'quarter_4']
số dòng của bảng dài = 2 x 4 = 8


### 3.2. Danh sách thành phần (list comprehension) để chọn cột

`[c for c in report.columns if c != "product"]` đọc là "lấy mọi cột `c` của bảng mà `c` khác `product`". Cách viết ngắn của vòng `for` + `if` + `append`. Ta dùng nó để tự xác định các cột giá trị.

In [3]:
cols = list(report.columns)
print(cols)
others = []
for c in cols:                    # cách viết dài
    if c != "product":
        others.append(c)
print(others == [c for c in cols if c != "product"])      # True: hai cách cho cùng kết quả

['product', 'quarter_1', 'quarter_2', 'quarter_3', 'quarter_4']
True


### 3.3. Hai vòng lặp lồng nhau và `zip`

Để tạo bảng dài bằng tay ta cần **hai vòng lặp lồng nhau**: vòng **ngoài** duyệt từng **cột giá trị** (từng quý), vòng **trong** duyệt từng **hàng** (từng sản phẩm). `zip(a, b)` ghép hai dãy cùng độ dài từng cặp; `report[q]` lấy cột `q`.

In [4]:
for q in ["quarter_1", "quarter_2"]:                     # vòng ngoài: từng quý
    for product, sales in zip(report["product"], report[q]):   # vòng trong: từng sản phẩm
        print(q, product, sales)

quarter_1 Umbrella 417
quarter_1 SleepingBag 800
quarter_2 Umbrella 224
quarter_2 SleepingBag 936


### 3.4. `DataFrame.melt(id_vars, value_vars, var_name, value_name)`

- `id_vars="product"`: cột **giữ nguyên** (định danh), lặp lại trên mỗi dòng mới;
- `value_vars=[...]`: các cột **bị gập xuống** (mặc định: mọi cột còn lại không thuộc `id_vars`);
- `var_name="quarter"`: tên cột chứa **tên các cột bị gập** (mặc định `variable`);
- `value_name="sales"`: tên cột chứa **giá trị** (mặc định `value`).

Kết quả là **bảng mới**, nhãn hàng `0..n-1`.

In [5]:
long = report.melt(id_vars="product", var_name="quarter", value_name="sales")
print(long)
print(long.shape)

       product    quarter  sales
0     Umbrella  quarter_1    417
1  SleepingBag  quarter_1    800
2     Umbrella  quarter_2    224
3  SleepingBag  quarter_2    936
4     Umbrella  quarter_3    379
5  SleepingBag  quarter_3     93
6     Umbrella  quarter_4    611
7  SleepingBag  quarter_4    875
(8, 3)


### 3.5. Tên mặc định `variable` và `value`

Nếu không đặt `var_name` và `value_name`, pandas dùng **`variable`** và **`value`**. Đề yêu cầu tên cột `quarter` và `sales`, nên **bắt buộc** đặt hai tham số này (hoặc đổi tên sau bằng `rename`).

In [6]:
default = report.melt(id_vars="product")
print(default.columns.tolist())                       # ['product', 'variable', 'value']
print(default.rename(columns={"variable": "quarter", "value": "sales"}).columns.tolist())

['product', 'variable', 'value']
['product', 'quarter', 'sales']


### 3.6. `stack()` — gập các cột xuống index

`df.set_index("product").stack()` đưa **mọi cột còn lại** thành **cấp thứ hai của index**, kết quả là một **Series** có index hai cấp (`product`, tên cột). `reset_index(name="sales")` biến index về các cột thường. Khác `melt` ở **thứ tự dòng**: `stack` đi **từng hàng một** (Umbrella q1..q4 rồi SleepingBag q1..q4).

In [7]:
st = report.set_index("product").stack()
print(st.head(5))
print(st.index.names)                               # ['product', None]
flat = st.reset_index()
print(flat.columns.tolist(), flat.shape)           # cột thứ hai tên level_1 (chưa đặt tên)

product               
Umbrella     quarter_1    417
             quarter_2    224
             quarter_3    379
             quarter_4    611
SleepingBag  quarter_1    800
dtype: int64
['product', None]
['product', 'level_1', 0] (8, 3)


### 3.7. `assign` và `concat` để ghép các mảnh

`df.assign(cột=giá_trị)` thêm/ghi đè một cột và trả bảng mới. `set_axis([...], axis=1)` đặt lại tên cột. Ta có thể tạo **từng mảnh** `(product, quarter_i, sales)` rồi `pd.concat` các mảnh theo chiều dọc. Đó là cách "melt thủ công bằng pandas" (Cách 5).

In [8]:
piece = report[["product", "quarter_1"]].set_axis(["product", "sales"], axis=1).assign(quarter="quarter_1")
print(piece)
print(piece[["product", "quarter", "sales"]].columns.tolist())       # chọn lại thứ tự cột

       product  sales    quarter
0     Umbrella    417  quarter_1
1  SleepingBag    800  quarter_1
['product', 'quarter', 'sales']


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Hãy tưởng tượng bảng 2 hàng x 4 cột doanh số. Ta **đọc lần lượt từng cột doanh số**, và với **mỗi ô** viết ra **một dòng mới**: "sản phẩm - tên cột - số".

| bước | cột đang đọc | ô | dòng mới (product, quarter, sales) |
|---|---|---|---|
| 1 | quarter_1 | Umbrella | (Umbrella, quarter_1, 417) |
| 2 | quarter_1 | SleepingBag | (SleepingBag, quarter_1, 800) |
| 3 | quarter_2 | Umbrella | (Umbrella, quarter_2, 224) |
| 4 | quarter_2 | SleepingBag | (SleepingBag, quarter_2, 936) |
| ... | ... | ... | ... |
| 8 | quarter_4 | SleepingBag | (SleepingBag, quarter_4, 875) |

Thuật toán: **với mỗi cột giá trị, với mỗi hàng, sinh một dòng**. Tổng `4 x 2 = 8` dòng, đúng thứ tự trong đề.

## 5. Cách 1 — Brute Force / baseline: hai vòng lặp lồng nhau

**Ý tưởng.** Vòng ngoài duyệt các cột giá trị, vòng trong duyệt các hàng; mỗi ô sinh một từ điển `{"product":..., "quarter":..., "sales":...}`. Cuối cùng dựng bảng từ danh sách đó.

In [9]:
def melt_loop(report: pd.DataFrame) -> pd.DataFrame:
    value_cols = [c for c in report.columns if c != "product"]     # các cột sẽ gập xuống
    rows = []
    for q in value_cols:                                           # vòng ngoài: từng quý
        for product, sales in zip(report["product"], report[q]):   # vòng trong: từng sản phẩm
            rows.append({"product": product, "quarter": q, "sales": sales})
    out = pd.DataFrame(rows, columns=["product", "quarter", "sales"])
    return out.astype({"product": report["product"].dtype, "quarter": "str", "sales": "int64"})   # chốt kiểu cột


print(melt_loop(report))

       product    quarter  sales
0     Umbrella  quarter_1    417
1  SleepingBag  quarter_1    800
2     Umbrella  quarter_2    224
3  SleepingBag  quarter_2    936
4     Umbrella  quarter_3    379
5  SleepingBag  quarter_3     93
6     Umbrella  quarter_4    611
7  SleepingBag  quarter_4    875


In [10]:
res = melt_loop(report)
print("hình dạng:", res.shape)                           # (8, 3)
print("quarter  :", res["quarter"].tolist())
print("sales    :", res["sales"].tolist())
print("dtype    :", res.dtypes.to_dict())

hình dạng: (8, 3)
quarter  : ['quarter_1', 'quarter_1', 'quarter_2', 'quarter_2', 'quarter_3', 'quarter_3', 'quarter_4', 'quarter_4']
sales    : [417, 800, 224, 936, 379, 93, 611, 875]
dtype    : {'product': <StringDtype(storage='python', na_value=nan)>, 'quarter': <StringDtype(storage='python', na_value=nan)>, 'sales': dtype('int64')}


**Độ phức tạp (Cách 1).** Gọi `m` là số hàng, `q` là số cột giá trị. Hai vòng lồng tạo `m·q` dòng, mỗi dòng `O(1)`: **Time `O(m·q)`**; danh sách `rows` giữ `m·q` từ điển: **Space `O(m·q)`**.

**Nút thắt.** Tạo hàng triệu từ điển Python rất chậm và tốn bộ nhớ; phải tự chốt kiểu cột. pandas có `melt` làm bằng mã C.

## 6. Ý tưởng thuật toán / bước đột phá

pandas gói đúng thao tác ta vừa làm bằng tay:

`report.melt(id_vars="product", var_name="quarter", value_name="sales")`

- `id_vars="product"`: cột giữ nguyên, được **lặp lại** cho từng dòng;
- các cột còn lại tự động là `value_vars`, bị **gập** xuống: tên cột vào `quarter`, giá trị vào `sales`;
- thứ tự dòng: **theo từng cột giá trị** (đúng như Cách 1);
- nhãn hàng `0..n-1`.

Các cách khác: nêu rõ **`value_vars`** (Cách 3), **`stack`** (Cách 4), **`concat` từng mảnh** (Cách 5).

## 7. Các cách tiếp theo

### Cách 2 — `melt(id_vars=..., var_name=..., value_name=...)` (cách chuẩn)

**Giải thích.** Một lời gọi, không sửa bảng đầu vào.

In [11]:
def melt_melt(report: pd.DataFrame) -> pd.DataFrame:
    return report.melt(id_vars="product", var_name="quarter", value_name="sales")


print(melt_melt(report))

       product    quarter  sales
0     Umbrella  quarter_1    417
1  SleepingBag  quarter_1    800
2     Umbrella  quarter_2    224
3  SleepingBag  quarter_2    936
4     Umbrella  quarter_3    379
5  SleepingBag  quarter_3     93
6     Umbrella  quarter_4    611
7  SleepingBag  quarter_4    875


In [12]:
# Code nộp LeetCode cho Cách 2
import pandas as pd

def meltTable(report: pd.DataFrame) -> pd.DataFrame:
    return report.melt(id_vars="product", var_name="quarter", value_name="sales")

**Độ phức tạp Cách 2:** **Time `O(m·q)`** (mỗi ô doanh số được chép đúng một lần), **Space `O(m·q)`** (bảng kết quả `m·q` dòng).

### Cách 3 — `melt` với `value_vars` tường minh

**Giải thích.** Liệt kê **rõ** các cột cần gập (`value_vars=[...]`) thay vì để pandas tự suy ra "mọi cột còn lại". An toàn hơn khi bảng có thêm cột khác mà ta không muốn gập (ví dụ `notes`, `year`). Cũng có thể dùng `pd.melt(report, ...)` (hàm tương đương).

In [13]:
def melt_value_vars(report: pd.DataFrame) -> pd.DataFrame:
    quarters = ["quarter_1", "quarter_2", "quarter_3", "quarter_4"]
    return pd.melt(report, id_vars=["product"], value_vars=quarters,
                   var_name="quarter", value_name="sales")


print(melt_value_vars(report))

       product    quarter  sales
0     Umbrella  quarter_1    417
1  SleepingBag  quarter_1    800
2     Umbrella  quarter_2    224
3  SleepingBag  quarter_2    936
4     Umbrella  quarter_3    379
5  SleepingBag  quarter_3     93
6     Umbrella  quarter_4    611
7  SleepingBag  quarter_4    875


In [14]:
# Code nộp LeetCode cho Cách 3
import pandas as pd

def meltTable(report: pd.DataFrame) -> pd.DataFrame:
    return pd.melt(report, id_vars=["product"],
                   value_vars=["quarter_1", "quarter_2", "quarter_3", "quarter_4"],
                   var_name="quarter", value_name="sales")

**Độ phức tạp Cách 3:** **Time `O(m·q)`**, **Space `O(m·q)`**.

### Cách 4 — `set_index("product").stack()` rồi `reset_index`

**Giải thích.** `stack` gập các cột xuống index; thứ tự dòng là **từng hàng một** (khác `melt`). Muốn khớp thứ tự với đề, sắp xếp ổn định theo cột `quarter` (`kind="stable"` giữ thứ tự sản phẩm trong mỗi quý).

In [15]:
def melt_stack(report: pd.DataFrame) -> pd.DataFrame:
    long = report.set_index("product").stack().reset_index()
    long.columns = ["product", "quarter", "sales"]
    return long.sort_values("quarter", kind="stable", ignore_index=True)   # đưa về thứ tự theo quý


raw = report.set_index("product").stack().reset_index()
print("stack (thứ tự theo hàng):", raw.iloc[:, 0].tolist())
print(melt_stack(report))

stack (thứ tự theo hàng): ['Umbrella', 'Umbrella', 'Umbrella', 'Umbrella', 'SleepingBag', 'SleepingBag', 'SleepingBag', 'SleepingBag']
       product    quarter  sales
0     Umbrella  quarter_1    417
1  SleepingBag  quarter_1    800
2     Umbrella  quarter_2    224
3  SleepingBag  quarter_2    936
4     Umbrella  quarter_3    379
5  SleepingBag  quarter_3     93
6     Umbrella  quarter_4    611
7  SleepingBag  quarter_4    875


**Độ phức tạp Cách 4:** **Time `O(m·q)`** cộng sắp xếp `O(m·q·log(m·q))` nếu cần đổi thứ tự, **Space `O(m·q)`**.

### Cách 5 — `concat` các mảnh theo từng cột giá trị

**Giải thích.** Với mỗi cột giá trị `q`, tạo một bảng nhỏ `(product, q)` đổi tên cột giá trị thành `sales`, thêm cột `quarter = q`; rồi `pd.concat` tất cả theo chiều dọc. Là phiên bản "đọc từng cột rồi nối" của Cách 1 nhưng dùng thao tác bảng.

In [16]:
def melt_concat(report: pd.DataFrame) -> pd.DataFrame:
    pieces = []
    for q in report.columns.drop("product"):
        piece = report[["product", q]].set_axis(["product", "sales"], axis=1).assign(quarter=q)
        pieces.append(piece[["product", "quarter", "sales"]])
    return pd.concat(pieces, ignore_index=True)


print(melt_concat(report))

       product    quarter  sales
0     Umbrella  quarter_1    417
1  SleepingBag  quarter_1    800
2     Umbrella  quarter_2    224
3  SleepingBag  quarter_2    936
4     Umbrella  quarter_3    379
5  SleepingBag  quarter_3     93
6     Umbrella  quarter_4    611
7  SleepingBag  quarter_4    875


**Độ phức tạp Cách 5:** **Time `O(m·q)`**, **Space `O(m·q)`**; chạy `q` vòng (số cột giá trị, thường nhỏ) nên nhanh hơn Cách 1 nhiều khi `m` lớn.

> Cách 2 là cách nên nộp: đặt đúng `var_name`, `value_name`.

## 8. TRACE từng bước

Ta theo dõi (a) hai vòng lặp tự viết: từng dòng sinh ra; (b) pipeline `melt`: dựng từng khối theo cột giá trị.

In [17]:
def trace_loop(report):
    value_cols = [c for c in report.columns if c != "product"]
    print("cột giá trị:", value_cols)
    rows = []
    step = 0
    for q in value_cols:
        for product, sales in zip(report["product"], report[q]):
            step += 1
            rows.append({"product": product, "quarter": q, "sales": sales})
            print(f"bước {step}: q={q}, product={product:<11} sales={sales:>3} -> thêm dòng (độ dài rows = {len(rows)})")
    out = pd.DataFrame(rows, columns=["product", "quarter", "sales"])
    print("\nbảng dài:", out.shape)
    print(out)
    return out


_ = trace_loop(report)

cột giá trị: ['quarter_1', 'quarter_2', 'quarter_3', 'quarter_4']
bước 1: q=quarter_1, product=Umbrella    sales=417 -> thêm dòng (độ dài rows = 1)
bước 2: q=quarter_1, product=SleepingBag sales=800 -> thêm dòng (độ dài rows = 2)
bước 3: q=quarter_2, product=Umbrella    sales=224 -> thêm dòng (độ dài rows = 3)
bước 4: q=quarter_2, product=SleepingBag sales=936 -> thêm dòng (độ dài rows = 4)
bước 5: q=quarter_3, product=Umbrella    sales=379 -> thêm dòng (độ dài rows = 5)
bước 6: q=quarter_3, product=SleepingBag sales= 93 -> thêm dòng (độ dài rows = 6)
bước 7: q=quarter_4, product=Umbrella    sales=611 -> thêm dòng (độ dài rows = 7)
bước 8: q=quarter_4, product=SleepingBag sales=875 -> thêm dòng (độ dài rows = 8)

bảng dài: (8, 3)
       product    quarter  sales
0     Umbrella  quarter_1    417
1  SleepingBag  quarter_1    800
2     Umbrella  quarter_2    224
3  SleepingBag  quarter_2    936
4     Umbrella  quarter_3    379
5  SleepingBag  quarter_3     93
6     Umbrella  quarter_4    

In [18]:
def trace_melt(report):
    print("BƯỚC 0 - bảng rộng", report.shape)
    print(report)

    id_vars = ["product"]
    value_vars = [c for c in report.columns if c not in id_vars]
    print("\nBƯỚC 1 - id_vars =", id_vars, "| value_vars =", value_vars)

    print("\nBƯỚC 2 - mỗi cột giá trị tạo một khối", len(report), "dòng:")
    for q in value_vars:
        print(f"   khối {q}: product={report['product'].tolist()} sales={report[q].tolist()}")

    out = report.melt(id_vars="product", var_name="quarter", value_name="sales")
    print("\nBƯỚC 3 - melt trả về:", out.shape)
    print(out)
    print("nhãn hàng:", out.index.tolist())
    return out


_ = trace_melt(report)

BƯỚC 0 - bảng rộng (2, 5)
       product  quarter_1  quarter_2  quarter_3  quarter_4
0     Umbrella        417        224        379        611
1  SleepingBag        800        936         93        875

BƯỚC 1 - id_vars = ['product'] | value_vars = ['quarter_1', 'quarter_2', 'quarter_3', 'quarter_4']

BƯỚC 2 - mỗi cột giá trị tạo một khối 2 dòng:
   khối quarter_1: product=['Umbrella', 'SleepingBag'] sales=[417, 800]
   khối quarter_2: product=['Umbrella', 'SleepingBag'] sales=[224, 936]
   khối quarter_3: product=['Umbrella', 'SleepingBag'] sales=[379, 93]
   khối quarter_4: product=['Umbrella', 'SleepingBag'] sales=[611, 875]

BƯỚC 3 - melt trả về: (8, 3)
       product    quarter  sales
0     Umbrella  quarter_1    417
1  SleepingBag  quarter_1    800
2     Umbrella  quarter_2    224
3  SleepingBag  quarter_2    936
4     Umbrella  quarter_3    379
5  SleepingBag  quarter_3     93
6     Umbrella  quarter_4    611
7  SleepingBag  quarter_4    875
nhãn hàng: [0, 1, 2, 3, 4, 5, 6, 7]


Bảng tóm tắt:

| bước | đối tượng | giá trị |
|---|---|---|
| 0 | bảng rộng | `(2, 5)` |
| 1 | `id_vars`, `value_vars` | `["product"]`, 4 cột quý |
| 2 | khối theo từng quý | 4 khối, mỗi khối 2 dòng |
| 3 | `melt` | bảng dài `(8, 3)`, nhãn `0..7` |

## 9. Code nộp LeetCode

Bài pandas không có `class Solution`: chỉ cần định nghĩa hàm đúng tên và chữ ký (đối chiếu lại với đề gốc).

In [19]:
import pandas as pd

def meltTable(report: pd.DataFrame) -> pd.DataFrame:
    return report.melt(id_vars="product", var_name="quarter", value_name="sales")


print(meltTable(report))

       product    quarter  sales
0     Umbrella  quarter_1    417
1  SleepingBag  quarter_1    800
2     Umbrella  quarter_2    224
3  SleepingBag  quarter_2    936
4     Umbrella  quarter_3    379
5  SleepingBag  quarter_3     93
6     Umbrella  quarter_4    611
7  SleepingBag  quarter_4    875


## 10. Vì sao thuật toán đúng

- **Mỗi ô doanh số thành đúng một dòng:** `melt` lấy mọi cột trong `value_vars` (ở đây 4 quý) và sinh một dòng cho **mỗi** cặp (hàng, cột giá trị); không bỏ ô nào, không nhân đôi ô nào. Do đó `len(result) = m · q`.
- **Đúng nội dung dòng:** dòng của ô (hàng `i`, cột `q`) mang `product` của hàng `i` (lặp lại từ `id_vars`), `quarter = q` (tên cột) và `sales =` giá trị ô.
- **Đúng thứ tự:** `melt` xếp theo từng cột giá trị theo thứ tự cột trong bảng gốc, trong mỗi cột theo thứ tự hàng; trùng với thứ tự trong đề.
- **Đúng tên cột:** `var_name="quarter"`, `value_name="sales"` đặt tên như yêu cầu; thứ tự cột kết quả: `id_vars` trước, rồi `quarter`, `sales`.
- **Tính chất kiểm được:** với mọi `i`, `q`: `result` có đúng một dòng `(product_i, q, report.loc[i, q])`; và `result["sales"].sum() == report[value_cols].to_numpy().sum()`.

## 11. Phân tích Time / Space Complexity

Gọi `m` là số hàng của bảng rộng, `q` là số cột giá trị (số quý). Bảng dài có `N = m·q` dòng.

- **Time:** `O(N)`: mỗi ô doanh số được chép đúng một lần. `melt` chạy bằng mã C (hằng số nhỏ); hai vòng lặp Python hằng số lớn hơn nhiều nhưng cùng bậc.
- **Space:** `O(N)` cho bảng dài (cột `product` được lặp lại `q` lần).
- Không thể nhanh hơn `O(N)` vì kết quả có `N` dòng.

## 12. Bộ test

Ta đưa các cách vào từ điển `impls`. Các hàm không sửa bảng đầu vào.

In [20]:
impls = {
    "loop": melt_loop,
    "melt": melt_melt,
    "value_vars": melt_value_vars,
    "stack": melt_stack,
    "concat": melt_concat,
    "submit": meltTable,
}

expected_example = pd.DataFrame({
    "product": ["Umbrella", "SleepingBag"] * 4,
    "quarter": ["quarter_1"] * 2 + ["quarter_2"] * 2 + ["quarter_3"] * 2 + ["quarter_4"] * 2,
    "sales": [417, 800, 224, 936, 379, 93, 611, 875],
})

for key, fn in impls.items():
    pd.testing.assert_frame_equal(fn(report), expected_example)
print("PASS ví dụ của đề (mọi cách)")

PASS ví dụ của đề (mọi cách)


### 12.1. Các trường hợp biên

In [21]:
def make(products, table):
    """table[i] = doanh số 4 quý của sản phẩm thứ i."""
    data = {"product": pd.Series(products, dtype="str")}
    for j in range(4):
        data[f"quarter_{j + 1}"] = pd.Series([row[j] for row in table], dtype="int64")
    return pd.DataFrame(data)


def check_all(df, label):
    n = len(df)
    for key, fn in impls.items():
        out = fn(df)
        assert out.shape == (4 * n, 3), (label, key, out.shape)
        assert out.columns.tolist() == ["product", "quarter", "sales"], (label, key)
        # mỗi ô gốc xuất hiện đúng một lần với đúng giá trị
        for i in range(n):
            for j in range(4):
                row = out.iloc[j * n + i]                 # thứ tự: từng quý, trong mỗi quý theo hàng
                assert row["product"] == df["product"].iloc[i], (label, key, i, j)
                assert row["quarter"] == f"quarter_{j + 1}", (label, key, i, j)
                assert row["sales"] == df[f"quarter_{j + 1}"].iloc[i], (label, key, i, j)
    print("PASS", label, "->", 4 * n, "dòng")


check_all(make(["A"], [[1, 2, 3, 4]]), "một sản phẩm")
check_all(make(["A", "B", "C"], [[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12]]), "ba sản phẩm")
check_all(make(["A", "B"], [[0, 0, 0, 0], [0, 0, 0, 0]]), "toàn doanh số 0")
check_all(make(["A", "B"], [[7, 7, 7, 7], [7, 7, 7, 7]]), "doanh số bằng nhau")
check_all(make(["A", "A"], [[1, 2, 3, 4], [1, 2, 3, 4]]), "tên sản phẩm trùng nhau")

PASS một sản phẩm -> 4 dòng
PASS ba sản phẩm -> 12 dòng
PASS toàn doanh số 0 -> 8 dòng
PASS doanh số bằng nhau -> 8 dòng
PASS tên sản phẩm trùng nhau -> 8 dòng


### 12.2. Bảng rỗng, tính chất, đầu vào, và phản ví dụ

In [22]:
# Bảng rỗng: kết quả rỗng nhưng vẫn đủ 3 cột
empty = make([], [])
for key, fn in impls.items():
    out = fn(empty)
    assert len(out) == 0 and out.columns.tolist() == ["product", "quarter", "sales"], key
print("PASS bảng rỗng: kết quả rỗng, đủ 3 cột")

# Tính chất: tổng doanh số được bảo toàn, mỗi quý đủ số dòng
out = meltTable(report)
assert out["sales"].sum() == report[["quarter_1", "quarter_2", "quarter_3", "quarter_4"]].to_numpy().sum()
assert out["quarter"].value_counts().to_dict() == {"quarter_1": 2, "quarter_2": 2, "quarter_3": 2, "quarter_4": 2}
print("PASS tổng doanh số được bảo toàn; mỗi quý có đủ 2 dòng")

# Đảo ngược melt bằng pivot trở lại bảng rộng
back = out.pivot(index="product", columns="quarter", values="sales").reindex(report["product"])
assert back["quarter_3"].tolist() == [379, 93]
print("PASS pivot đảo lại được melt")

# Đầu vào không bị sửa
before = report.copy()
for key, fn in impls.items():
    fn(report)
pd.testing.assert_frame_equal(report, before)
print("PASS đầu vào không bị sửa")

# Phản ví dụ 1: quên var_name / value_name -> sai tên cột
wrong = report.melt(id_vars="product")
assert wrong.columns.tolist() == ["product", "variable", "value"]
print("PASS phản ví dụ: không đặt tên ->", wrong.columns.tolist())

# Phản ví dụ 2: quên id_vars -> melt luôn cả product, mất cột product
wrong2 = report.melt(var_name="quarter", value_name="sales")
assert wrong2.shape == (10, 2)      # 2 + 8 giá trị, product bị gập lẫn vào cột sales
print("PASS phản ví dụ: không có id_vars ->", wrong2.shape, wrong2.columns.tolist())

PASS bảng rỗng: kết quả rỗng, đủ 3 cột
PASS tổng doanh số được bảo toàn; mỗi quý có đủ 2 dòng
PASS pivot đảo lại được melt
PASS đầu vào không bị sửa
PASS phản ví dụ: không đặt tên -> ['product', 'variable', 'value']
PASS phản ví dụ: không có id_vars -> (10, 2) ['quarter', 'sales']


### 12.3. Đối chiếu ngẫu nhiên (có seed)

In [23]:
import random

random.seed(2890)
for trial in range(200):
    n = random.randint(0, 8)
    names = [random.choice(["Umbrella", "Bag", "Tent", "Lamp"]) for _ in range(n)]
    table = [[random.randint(0, 1000) for _ in range(4)] for _ in range(n)]
    df = make(names, table)
    ref = df.melt(id_vars="product", var_name="quarter", value_name="sales")
    assert len(ref) == 4 * n
    for key, fn in impls.items():
        pd.testing.assert_frame_equal(fn(df), ref)
print("PASS 200 bảng ngẫu nhiên - mọi cách khớp nhau")

PASS 200 bảng ngẫu nhiên - mọi cách khớp nhau


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Quên đặt `var_name` và `value_name`

In [24]:
sai = report.melt(id_vars="product")
print("sai  :", sai.columns.tolist())              # ['product', 'variable', 'value']
dung = report.melt(id_vars="product", var_name="quarter", value_name="sales")
print("đúng :", dung.columns.tolist())

sai  : ['product', 'variable', 'value']
đúng : ['product', 'quarter', 'sales']


**Vì sao sai:** nếu không đặt, pandas dùng tên mặc định `variable` và `value`. Đề yêu cầu `quarter` và `sales`; sai tên cột là sai kết quả dù dữ liệu đúng.

### Lỗi 2: Quên `id_vars` — cột `product` bị gập luôn

In [25]:
sai = report.melt(var_name="quarter", value_name="sales")
print("sai  :", sai.shape, sai.columns.tolist())
print(sai.head(3))

sai  : (10, 2) ['quarter', 'sales']
     quarter        sales
0    product     Umbrella
1    product  SleepingBag
2  quarter_1          417


**Vì sao sai:** không có `id_vars`, **mọi cột** (cả `product`) được gập xuống; kết quả mất cột `product` (tên sản phẩm lẫn vào cột `sales` thành giá trị chữ). Luôn khai báo `id_vars="product"`.

### Lỗi 3: Nhầm `id_vars` với `value_vars`

In [26]:
sai = report.melt(id_vars=["quarter_1", "quarter_2", "quarter_3", "quarter_4"], value_vars="product")
print("sai  :", sai.shape, sai.columns.tolist())
dung = report.melt(id_vars="product", value_vars=["quarter_1", "quarter_2", "quarter_3", "quarter_4"])
print("đúng :", dung.shape)

sai  : (2, 6) ['quarter_1', 'quarter_2', 'quarter_3', 'quarter_4', 'variable', 'value']
đúng : (8, 3)


**Vì sao sai:** `id_vars` là cột **giữ nguyên** (định danh), `value_vars` là cột **bị gập**. Đảo hai vai trò sẽ gập cột `product` và giữ 4 cột quý, cho bảng sai hoàn toàn. Mẹo nhớ: "id" = "ai là ai" (định danh), "value" = "các giá trị".

### Lỗi 4: Quên rằng `stack` có thứ tự dòng khác `melt`

In [27]:
via_stack = report.set_index("product").stack().reset_index()
via_stack.columns = ["product", "quarter", "sales"]
print("stack:", list(zip(via_stack["product"], via_stack["quarter"]))[:3])
print("melt :", list(zip(meltTable(report)["product"], meltTable(report)["quarter"]))[:3])
print("giống nhau (không sắp xếp)?", via_stack.equals(meltTable(report)))

stack: [('Umbrella', 'quarter_1'), ('Umbrella', 'quarter_2'), ('Umbrella', 'quarter_3')]
melt : [('Umbrella', 'quarter_1'), ('SleepingBag', 'quarter_1'), ('Umbrella', 'quarter_2')]
giống nhau (không sắp xếp)? False


**Vì sao sai:** `stack` đi **từng hàng một** còn `melt` đi **từng cột một**. Nếu trình chấm so thứ tự dòng, dùng `stack` cần sắp lại (như Cách 4). Với đề này dùng `melt` cho đúng thứ tự.

### Lỗi 5: Nhầm hàm `melt` với `pivot`

In [28]:
long = meltTable(report)
print("melt (rộng -> dài):", report.shape, "->", long.shape)
wide = long.pivot(index="product", columns="quarter", values="sales")
print("pivot (dài -> rộng):", long.shape, "->", wide.shape)

melt (rộng -> dài): (2, 5) -> (8, 3)
pivot (dài -> rộng): (8, 3) -> (2, 4)


**Vì sao dễ nhầm:** `melt` làm bảng **dài hơn** (thêm dòng, bớt cột), `pivot` làm bảng **rộng hơn**. Đề bài cho bảng rộng và muốn bảng dài -> `melt`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Thấy "gập các cột thành dòng / unpivot / tên cột là giá trị / wide to long" → nghĩ tới **`melt`**.
- Công thức nhớ: **`df.melt(id_vars="cột giữ", var_name="tên cho cột-tên", value_name="tên cho cột-giá-trị")`**.
- Đếm kiểm: số dòng mới = `số hàng x số cột giá trị`.
- Muốn **chỉ gập một số cột**: dùng `value_vars=[...]`.
- Dài -> rộng là `pivot` / `pivot_table`; rộng -> dài là `melt`; thêm hậu tố số cột thì `pd.wide_to_long`.
- Sau `melt` có thể cần `sort_values([...])` hoặc `reset_index(drop=True)`; đối chiếu đề.
- Luôn in `df.shape` trước và sau để kiểm tra nhanh.

## 15. Biến thể của bài

1. **Chỉ gập hai quý**: `report.melt(id_vars="product", value_vars=["quarter_1", "quarter_2"], var_name="quarter", value_name="sales")`.
2. **Giữ nhiều cột định danh**: `df.melt(id_vars=["product", "year"], ...)`.
3. Sau melt, **bỏ chữ `quarter_`** để lấy số quý: `long["quarter"].str.replace("quarter_", "").astype(int)`.
4. **Sắp xếp** theo sản phẩm rồi quý: `long.sort_values(["product", "quarter"])`.
5. **Tổng doanh số mỗi sản phẩm** (từ bảng dài): `long.groupby("product")["sales"].sum()`.
6. **Đảo lại** (pivot): `long.pivot(index="product", columns="quarter", values="sales")`.

Ta thử vài biến thể:

In [29]:
long = meltTable(report)
print("1.", report.melt(id_vars="product", value_vars=["quarter_1", "quarter_2"], var_name="quarter", value_name="sales").shape)

num = long.assign(quarter=long["quarter"].str.replace("quarter_", "", regex=False).astype(int))
print("3. số quý :", num["quarter"].tolist())
print("4. sắp xếp:", long.sort_values(["product", "quarter"])["product"].tolist()[:4])
print("5. tổng   :", long.groupby("product")["sales"].sum().to_dict())
print("6. pivot  :", long.pivot(index="product", columns="quarter", values="sales").shape)

1. (4, 3)
3. số quý : [1, 1, 2, 2, 3, 3, 4, 4]
4. sắp xếp: ['SleepingBag', 'SleepingBag', 'SleepingBag', 'SleepingBag']
5. tổng   : {'SleepingBag': 2704, 'Umbrella': 1631}
6. pivot  : (2, 4)


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Hai vòng lặp + danh sách từ điển | tự sinh từng dòng | `O(m·q)` (Python, chậm) | `O(m·q)` | hiểu bản chất |
| 2. `melt(id_vars, var_name, value_name)` | gập cột thành dòng | `O(m·q)` | `O(m·q)` | **cách nộp bài** |
| 3. `pd.melt(..., value_vars=[...])` | chỉ rõ cột cần gập | `O(m·q)` | `O(m·q)` | bảng có thêm cột không muốn gập |
| 4. `set_index().stack().reset_index()` | gập xuống index, thứ tự theo hàng | `O(m·q)` + sắp xếp | `O(m·q)` | dữ liệu đã ở dạng index; cần sắp lại thứ tự |
| 5. `concat` các mảnh theo cột | nối từng cột con | `O(m·q)` | `O(m·q)` | hiểu "melt = nối các cột"; tùy biến từng mảnh |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `report.melt(id_vars="product").shape`, `report.melt(id_vars="product").columns.tolist()`, `meltTable(report)["quarter"].nunique()`, `meltTable(report)["sales"].sum()`, `meltTable(report).iloc[1].tolist()`.

**Bài 2 (tính tay).** Bảng rộng: `product=[A, B]`, `x=[1, 2]`, `y=[3, 4]`. Viết bảng dài của `melt(id_vars="product", var_name="k", value_name="v")` (4 dòng), theo đúng thứ tự `melt`.

**Bài 3 (điền TODO).** Hoàn thành `melt_todo(df)` bằng `melt`.

In [30]:
def melt_todo(df):
    # TODO: return df.melt(id_vars=..., var_name=..., value_name=...)
    # gợi ý: giữ cột product; tên cột mới là quarter và sales
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi.

**Bài 4 (tự viết lại).** Viết `melt_loop2(report)` bằng hai vòng lặp lồng nhau (Cách 1) mà không nhìn code mẫu. Kiểm tra với ví dụ của đề.

**Bài 5 (biến thể / mini challenge).** (a) Viết `melt_quarter_number(report)` trả về bảng dài có cột `quarter` là **số nguyên** `1..4` (thay vì `"quarter_1"`). (b) Viết `best_quarter(report)` trả `dict` {sản phẩm: số quý có doanh số cao nhất} dựa trên bảng dài.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `(8, 3)`, `['product', 'variable', 'value']`, `4`, `4335` (tính tổng: `417+224+379+611+800+936+93+875`), dòng thứ 2 là `['SleepingBag', 'quarter_1', 800]`. Bài 2: mỗi cột `x`, `y` tạo một khối 2 dòng.
- **Mức 2 (vừa):** Bài 2: `(A, x, 1)`, `(B, x, 2)`, `(A, y, 3)`, `(B, y, 4)`. Bài 3: `id_vars="product"`, `var_name="quarter"`, `value_name="sales"`. Bài 4: vòng ngoài duyệt cột giá trị, vòng trong `zip(report["product"], report[q])`.
- **Mức 3 (sâu):** Bài 5(a): `long = meltTable(report)`, rồi `long["quarter"] = long["quarter"].str.replace("quarter_", "", regex=False).astype(int)`. Bài 5(b): `long.loc[long.groupby("product")["sales"].idxmax()]` cho dòng tốt nhất của mỗi sản phẩm, rồi lấy cột `quarter`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [31]:
# Bài 1
assert report.melt(id_vars="product").shape == (8, 3)
assert report.melt(id_vars="product").columns.tolist() == ["product", "variable", "value"]
assert meltTable(report)["quarter"].nunique() == 4
assert meltTable(report)["sales"].sum() == 4335
assert meltTable(report).iloc[1].tolist() == ["SleepingBag", "quarter_1", 800]

# Bài 2
t2 = pd.DataFrame({"product": ["A", "B"], "x": [1, 2], "y": [3, 4]})
l2 = t2.melt(id_vars="product", var_name="k", value_name="v")
assert list(zip(l2["product"], l2["k"], l2["v"])) == [("A", "x", 1), ("B", "x", 2), ("A", "y", 3), ("B", "y", 4)]

# Bài 3
def melt_todo_ans(df):
    return df.melt(id_vars="product", var_name="quarter", value_name="sales")

pd.testing.assert_frame_equal(melt_todo_ans(report), expected_example)

# Bài 4
def melt_loop2(report):
    value_cols = [c for c in report.columns if c != "product"]
    rows = []
    for q in value_cols:
        for product, sales in zip(report["product"], report[q]):
            rows.append({"product": product, "quarter": q, "sales": sales})
    return pd.DataFrame(rows, columns=["product", "quarter", "sales"]).astype({"product": report["product"].dtype, "quarter": "str", "sales": "int64"})

pd.testing.assert_frame_equal(melt_loop2(report), expected_example)

# Bài 5(a)
def melt_quarter_number(report):
    long = meltTable(report)
    long["quarter"] = long["quarter"].str.replace("quarter_", "", regex=False).astype(int)
    return long

assert melt_quarter_number(report)["quarter"].tolist() == [1, 1, 2, 2, 3, 3, 4, 4]

# Bài 5(b)
def best_quarter(report):
    long = melt_quarter_number(report)
    best = long.loc[long.groupby("product")["sales"].idxmax()]
    return {p: int(q) for p, q in zip(best["product"], best["quarter"])}

assert best_quarter(report) == {"Umbrella": 4, "SleepingBag": 2}
print("PASS đáp án phần 19")

PASS đáp án phần 19


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (gập các cột thành dòng)

```python
import pandas as pd

def meltTable(report: pd.DataFrame) -> pd.DataFrame:
    return report.melt(id_vars="product", var_name="quarter", value_name="sales")
```

**Checklist**

- [ ] `id_vars="product"` (cột giữ nguyên) và các cột quý bị gập (mặc định hoặc `value_vars=[...]`)?
- [ ] Đặt `var_name="quarter"` và `value_name="sales"` đúng tên đề yêu cầu?
- [ ] Số dòng kết quả = `số hàng x số quý` (ở ví dụ là 8)?
- [ ] Thứ tự dòng: từng quý một, trong mỗi quý theo thứ tự sản phẩm (không nhầm với `stack`)?
- [ ] **Trả về** DataFrame; bảng đầu vào không bị sửa?
- [ ] Phân biệt `melt` (rộng -> dài) và `pivot` (dài -> rộng)?
- [ ] Đã thử bảng một sản phẩm và bảng rỗng?
- [ ] Skeleton đúng `def meltTable(report: pd.DataFrame) -> pd.DataFrame` (đối chiếu lại với đề gốc nếu lệch)?